# Конструювання підказок | Prompt Engineering

In [32]:
import os
import openai
from dotenv import load_dotenv

load_dotenv()

key = os.getenv('OPEN_API_KEY')
client = openai.OpenAI(api_key=key)

# helper function
def get_completion(prompt, model="gpt-4o"): 
    response = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.7  # Adjust creativity
        )
    return response.choices[0].message.content

**Basics of prompt engineering.**

In [4]:
text = f"""
wait client.connect();
    const database = client.db("your-database-name");
    const collection = database.collection("your-collection-name");

    // Example: Insert a document
    const result = await collection.insertOne();

    // Example: Retrieve documents
    const documents = await collection.find().toArray();
"""

prompt = f"""
Summarize the code delimited by triple backticks \ 
into a paragraph.
Explain it to me like I am 5 years old
```{text}```
"""

response = get_completion(prompt)
print(response)

Alright, imagine you have a big box of toys (that's the "database") and each toy is in its own little box (that's the "collection"). First, we need to get into the big toy box, so we "connect" to it. Once we're in, we can do things like add a new toy to the little boxes inside (that's when we "insert a document"). We can also look at all the toys we have by opening up each little box and seeing what's inside (that's when we "retrieve documents" and turn them into a list with "toArray"). It's like playing with your toy collection, adding new toys, and looking at all the toys you have!


**Інструкція**

Ви можете створювати ефективні підказки для різних простих завдань, використовуючи команди, щоб вказати моделі, чого ви хочете досягти, наприклад «Написати», «Класифікувати», «Підсумувати», «Перекласти», «Впорядкувати» тощо.

Майте на увазі, що вам також потрібно багато експериментувати, щоб побачити, що працює найкраще. Спробуйте різні інструкції з різними ключовими словами, контекстами та даними і подивіться, що найкраще підходить для вашого конкретного випадку використання та завдання. Зазвичай, чим конкретнішим і релевантнішим є контекст для завдання, яке ви намагаєтеся виконати, тим краще. Ми торкнемося важливості вибірки та додавання більше контексту в наступних посібниках.

Інші рекомендують розміщувати інструкції на початку підказки. Ще одна рекомендація — використовувати чіткий роздільник, наприклад «###», щоб відокремити інструкцію від контексту.

In [ ]:
# prompt = f"""
# ### Instruction ###
# Translate the text below to Japanese:
# Text: "hello!"
# """
# Tone translation

prompt = f"""
### Instruction ###
Translate the following from slang to a business letter: 
Text: 'Dude, This is Joe, check out this spec on this standing lamp.'
"""
response = get_completion(prompt)
print(response)


Subject: Introduction to New Standing Lamp Specification

Dear [Recipient's Name],

I hope this message finds you well. My name is Joe, and I would like to bring to your attention the specifications of a new standing lamp that we believe may be of interest to you.

Please find attached the detailed specifications for your review. Should you have any questions or require further information, feel free to reach out.

Thank you for your time and consideration.

Best regards,

Joe


**Zero-Shot Prompting**

Сучасні великі мовні моделі (LLM), такі як GPT-3.5 Turbo, GPT-4 і Claude 3, налаштовані на виконання інструкцій і навчені на великих обсягах даних. Завдяки масштабному навчанню ці моделі здатні виконувати деякі завдання в режимі «zero-shot». Zero-shot prompting означає, що підказка, яка використовується для взаємодії з моделлю, не міститиме прикладів або демонстрацій. Zero-shot prompt безпосередньо дає моделі вказівку виконати завдання без додаткових прикладів для її керування.

In [10]:
prompt = f"""
Classify the text into neutral, negative or positive. 
Text: `I hate everything `
Sentiment:
"""

response = get_completion(prompt)
print(response)

Negative


**Few-Shot Prompting**

In [15]:
prompt = f"""
A "whatpu" is a small, furry animal native to Tanzania. An example of a sentence that uses the word whatpu is:
We were traveling in Africa and we saw these very cute whatpus.
 
To do a "farduddle" means to jump up and down really fast. An example of a sentence that uses the word farduddle is:
"""

response = get_completion(prompt)
print(response)

The children began to farduddle with excitement when they heard the ice cream truck's music playing in the distance.


**Chain-of-Thought Prompting**

Introduced in Wei et al. (2022), chain-of-thought (CoT) prompting enables complex reasoning capabilities through intermediate reasoning steps. You can combine it with few-shot prompting to get better results on more complex tasks that require reasoning before responding.

In [21]:
# Zero-shot COT Prompting - adding "Let's think step by step" to the original prompt
# prompt = "Скільки букв `р` в слові `Абракадабра`?"
# prompt += " Давай думати крок за кроком."

# response = get_completion(prompt)
# print(response)

# Proper Chain-of-Thought Prompting - few shot CoT prompting 
prompt = """
Скільки букв `д` в слові `Абракадабра`?
Відповідь: одна `д` на 7-й позиції.
Скільки букв `б` в слові `Абракадабра?
Відповідь: дві `б` на 2-й та на 9-й позиціях.
Скільки букв `р` в слові `Абракадабра`?"""
response = get_completion(prompt)
print(response)

У слові "Абракадабра" дві букви "р" на 3-й та 11-й позиціях.


**Дерево Думок (Tree of Thoughts, ToT)**

ToT підтримує дерево думок, де думки представляють собою послідовні мовні ланцюжки, які служать проміжними кроками на шляху до вирішення проблеми. Такий підхід дозволяє LM самостійно оцінювати прогрес, досягнутий за допомогою проміжних думок, на шляху до вирішення проблеми через обдуманий процес міркування. Здатність LM генерувати та оцінювати думки потім поєднується з алгоритмами пошуку (наприклад, пошук в ширину та пошук в глибину), щоб забезпечити систематичне дослідження думок з попереднім переглядом та поверненням назад.

In [ ]:
# question = "Скільки букв `р` в слові `Абракадабра`?"

# question = "The bottom of the mug was cut off and the top was sealed. Can you drink water from it?"

question = "I have to wash my car at the carwash. It is 50 meters away from my home. Should I drive there, or get there by walking?"


# response = get_completion(question)
# print(response)

prompt = f""" Imagine three different experts are answering this question.
All experts will write down 1 step of their thinking,
then share it with the group.
Then all experts will go on to the next step, etc.
If any expert realises they're wrong at any point then they leave.
The question is: ```{question}```"""
response = get_completion(prompt)
print(response)

If the carwash is only 50 meters away from your home, it would be more practical and environmentally friendly to walk there. Walking such a short distance is easy and saves you the hassle of getting in and out of your car, plus it avoids unnecessary fuel consumption and emissions.
